In [1]:
import pandas as pd

df = pd.read_csv('data.csv')

# Basic shape
print(df.shape)
print(df.columns.tolist())
print(df['Series Name'].unique())

# Countries
print(df['Country Name'].nunique(), 'countries')
print(df['Country Name'].unique()[:10])

# Missing value convention
print(df.iloc[0])  # Check what missing looks like

afg = df[df['Country Code'] == 'AFG']
print(afg[['Series Name', '2020 [YR2020]', '1960 [YR1960]']])

# Check missing values
year_cols = [c for c in df.columns if 'YR' in c]
missing = df[year_cols].isin(['..', '']).sum()
print(missing[missing > 0])

(44812, 95)
['Country Name', 'Country Code', 'Series Name', 'Series Code', '1960 [YR1960]', '1961 [YR1961]', '1962 [YR1962]', '1963 [YR1963]', '1964 [YR1964]', '1965 [YR1965]', '1966 [YR1966]', '1967 [YR1967]', '1968 [YR1968]', '1969 [YR1969]', '1970 [YR1970]', '1971 [YR1971]', '1972 [YR1972]', '1973 [YR1973]', '1974 [YR1974]', '1975 [YR1975]', '1976 [YR1976]', '1977 [YR1977]', '1978 [YR1978]', '1979 [YR1979]', '1980 [YR1980]', '1981 [YR1981]', '1982 [YR1982]', '1983 [YR1983]', '1984 [YR1984]', '1985 [YR1985]', '1986 [YR1986]', '1987 [YR1987]', '1988 [YR1988]', '1989 [YR1989]', '1990 [YR1990]', '1991 [YR1991]', '1992 [YR1992]', '1993 [YR1993]', '1994 [YR1994]', '1995 [YR1995]', '1996 [YR1996]', '1997 [YR1997]', '1998 [YR1998]', '1999 [YR1999]', '2000 [YR2000]', '2001 [YR2001]', '2002 [YR2002]', '2003 [YR2003]', '2004 [YR2004]', '2005 [YR2005]', '2006 [YR2006]', '2007 [YR2007]', '2008 [YR2008]', '2009 [YR2009]', '2010 [YR2010]', '2011 [YR2011]', '2012 [YR2012]', '2013 [YR2013]', '2014 [

In [2]:
print(df['Series Name'].unique())
print(df.groupby('Country Name').size().value_counts())

# Check missing value format
year_cols = [c for c in df.columns if 'YR' in c]
sample_vals = df[year_cols].values.flatten()
# Find non-numeric values
import numpy as np
non_numeric = set(v for v in sample_vals if not str(v).replace('.','').lstrip('-').isnumeric())
print('Non-numeric values found:', non_numeric)

['Age dependency ratio (% of working-age population)'
 'Age dependency ratio, old' 'Age dependency ratio, young'
 'Age population, age 0, female, interpolated'
 'Age population, age 0, male, interpolated'
 'Age population, age 01, female, interpolated'
 'Age population, age 01, male, interpolated'
 'Age population, age 02, female, interpolated'
 'Age population, age 02, male, interpolated'
 'Age population, age 03, female, interpolated'
 'Age population, age 03, male, interpolated'
 'Age population, age 04, female, interpolated'
 'Age population, age 04, male, interpolated'
 'Age population, age 05, female, interpolated'
 'Age population, age 05, male, interpolated'
 'Age population, age 06, female, interpolated'
 'Age population, age 06, male, interpolated'
 'Age population, age 07, female, interpolated'
 'Age population, age 07, male, interpolated'
 'Age population, age 08, female, interpolated'
 'Age population, age 08, male, interpolated'
 'Age population, age 09, female, interpola

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [3]:
import json
import math

# Replace NaN with None
def clean(val):
    if val is None:
        return None
    if isinstance(val, float) and math.isnan(val):
        return None
    return val

# Define age groups
age_groups = [
    '00-04', '05-09', '10-14', '15-19', '20-24', '25-29',
    '30-34', '35-39', '40-44', '45-49', '50-54', '55-59',
    '60-64', '65-69', '70-74', '75-79', '80+'
]

male_series   = [f'Male population {ag}' for ag in age_groups]
female_series = [f'Female population {ag}' for ag in age_groups]

year_cols = [c for c in df.columns if 'YR' in c]
years = [int(c.split(' ')[0]) for c in year_cols]

output = {}

for country_code, group in df.groupby('Country Code'):
    country_name = group['Country Name'].iloc[0]
    
    # Index by series name for easy lookup
    series_lookup = group.set_index('Series Name')
    
    # Skip if missing pyramid data
    if male_series[0] not in series_lookup.index:
        continue
    
    output[country_code] = {
        'name': country_name,
        'years': {}
    }
    
    for year, col in zip(years, year_cols):
        male_vals   = []
        female_vals = []
        
        for ms, fs in zip(male_series, female_series):
            m = series_lookup.loc[ms, col] if ms in series_lookup.index else None
            f = series_lookup.loc[fs, col] if fs in series_lookup.index else None
            
            # Convert '..'  to None
            male_vals.append(None if str(m) == '..' else float(m) if m else None)
            female_vals.append(None if str(f) == '..' else float(f) if f else None)
        
        # Total population for line graph
        total = None
        if 'Population, total' in series_lookup.index:
            raw = series_lookup.loc['Population, total', col]
            total = None if str(raw) == '..' else float(raw)
        
        output[country_code]['years'][year] = {
            'male':   [clean(v) for v in male_vals],
            'female': [clean(v) for v in female_vals],
            'total':  clean(total)
        }

# Save age group labels too
result = {
    'ageGroups': age_groups,
    'countries': output
}

with open('population.json', 'w') as f:
    json.dump(result, f)

print(f"Done! {len(output)} countries exported.")
print(f"Sample keys for AFG 1960: {list(output['AFG']['years'][1960].keys())}")
print(f"Male values AFG 1960: {output['AFG']['years'][1960]['male']}")

Done! 259 countries exported.
Sample keys for AFG 1960: ['male', 'female', 'total']
Male values AFG 1960: [779842.0, 600837.0, 524110.0, 476199.0, 415993.0, 361206.0, 319278.0, 267926.0, 222664.0, 183209.0, 147827.0, 122326.0, 93072.0, 67295.0, 40753.0, 19684.0, 8297.0]


In [8]:
world = df[df['Country Name'].str.contains('World', case=False, na=False)]
print(world[['Country Name', 'Country Code']].drop_duplicates())

      Country Name Country Code
1211    Arab World          ARB
44115        World          WLD


In [3]:
# Get world total population by year
world = df[df['Country Code'] == 'WLD']
total_pop = world[world['Series Name'] == 'Population, total']

year_cols = [c for c in df.columns if 'YR' in c]
years = [int(c.split(' ')[0]) for c in year_cols]

values = total_pop[year_cols].values.flatten()

for y, v in zip(years, values):
    if str(v) != '..':
        print(f"{y}: {float(v)/1e9:.3f}B")

1960: 3.034B
1961: 3.075B
1962: 3.128B
1963: 3.193B
1964: 3.258B
1965: 3.325B
1966: 3.395B
1967: 3.464B
1968: 3.535B
1969: 3.609B
1970: 3.685B
1971: 3.762B
1972: 3.839B
1973: 3.915B
1974: 3.991B
1975: 4.066B
1976: 4.139B
1977: 4.212B
1978: 4.286B
1979: 4.361B
1980: 4.438B
1981: 4.516B
1982: 4.597B
1983: 4.679B
1984: 4.760B
1985: 4.843B
1986: 4.929B
1987: 5.017B
1988: 5.106B
1989: 5.195B
1990: 5.285B
1991: 5.372B
1992: 5.456B
1993: 5.541B
1994: 5.625B
1995: 5.710B
1996: 5.793B
1997: 5.875B
1998: 5.957B
1999: 6.038B
2000: 6.118B
2001: 6.198B
2002: 6.277B
2003: 6.356B
2004: 6.436B
2005: 6.517B
2006: 6.598B
2007: 6.680B
2008: 6.764B
2009: 6.847B
2010: 6.931B
2011: 7.013B
2012: 7.097B
2013: 7.183B
2014: 7.269B
2015: 7.355B
2016: 7.442B
2017: 7.524B
2018: 7.606B
2019: 7.687B
2020: 7.767B
2021: 7.846B
2022: 7.924B
2023: 8.001B
2024: 8.077B
2025: 8.152B
2026: 8.226B
2027: 8.299B
2028: 8.371B
2029: 8.442B
2030: 8.512B
2031: 8.581B
2032: 8.649B
2033: 8.716B
2034: 8.783B
2035: 8.848B
2036: 8.913B

In [5]:
# Find child population change over year
year_cols = [c for c in df.columns if 'YR' in c]
years = [int(c.split(' ')[0]) for c in year_cols]

# Get 0-4 population (male + female)
male_0004   = df[df['Series Name'] == 'Male population 00-04'].copy()
female_0004 = df[df['Series Name'] == 'Female population 00-04'].copy()

output = {}

for _, male_row in male_0004.iterrows():
    code = male_row['Country Code']
    female_row = female_0004[female_0004['Country Code'] == code]
    if female_row.empty:
        continue

    output[code] = {}
    for year, col in zip(years, year_cols):
        m = male_row[col]
        f = female_row[col].values[0]
        if str(m) == '..' or str(f) == '..':
            output[code][year] = None
        else:
            output[code][year] = float(m) + float(f)

# Calculate year-over-year change
result = {}
for code, year_data in output.items():
    result[code] = {}
    sorted_years = sorted(year_data.keys())
    for i, year in enumerate(sorted_years):
        if i == 0 or year_data[year] is None:
            result[code][year] = None
            continue
        prev = year_data[sorted_years[i-1]]
        if prev is None or prev == 0:
            result[code][year] = None
            continue
        # Percentage change from previous year
        result[code][year] = (year_data[year] - prev) / prev * 100

with open('child_population_change.json', 'w') as f:
    json.dump(result, f)

print('Done!')
print(f'Sample AFG 1970: {result["AFG"][1970]:.2f}%')

Done!
Sample AFG 1970: 3.07%


In [6]:
# Clean the child population change data
def clean(val):
    if val is None:
        return None
    if isinstance(val, float) and math.isnan(val):
        return None
    return val

# Rebuild the output with cleaned values
result_clean = {}
for code, year_data in result.items():
    result_clean[code] = {}
    for year, val in year_data.items():
        result_clean[code][year] = clean(val)

with open('child_population_change.json', 'w') as f:
    json.dump(result_clean, f)

print('Done!')

Done!


In [2]:
# Find countries that have no Male population 00-04 data at all
male_0004 = df[df['Series Name'] == 'Male population 00-04']

no_data_countries = []
year_cols = [c for c in df.columns if 'YR' in c]

for _, row in male_0004.iterrows():
    vals = row[year_cols].values
    # Check if all values are missing
    all_missing = all(str(v) == '..' or str(v) == 'nan' for v in vals)
    if all_missing:
        no_data_countries.append({'code': row['Country Code'], 'name': row['Country Name']})

print(f"Countries with no data: {len(no_data_countries)}")
for c in no_data_countries:
    print(c['code'], c['name'])

Countries with no data: 24
ASM American Samoa
AND Andorra
BMU Bermuda
VGB British Virgin Islands
CYM Cayman Islands
DMA Dominica
FRO Faroe Islands
GIB Gibraltar
GRL Greenland
IMN Isle of Man
XKX Kosovo
LIE Liechtenstein
MHL Marshall Islands
MCO Monaco
NRU Nauru
MNP Northern Mariana Islands
INX Not classified
PLW Palau
SMR San Marino
SXM Sint Maarten (Dutch part)
KNA St. Kitts and Nevis
MAF St. Martin (French part)
TCA Turks and Caicos Islands
TUV Tuvalu
